In [2]:
# pubmed_fetcher.py
%pip install requests beautifulsoup4 lxml
import requests
from bs4 import BeautifulSoup

def fetch_pubmed_articles_with_metadata(query: str, max_results=3, use_mock_if_empty=True):
    headers = {"User-Agent": "Mozilla/5.0"}

    # Step 1: Search PubMed
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    search_params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json"
    }
    try:
        search_response = requests.get(search_url, params=search_params, headers=headers, timeout=10).json()
        id_list = search_response["esearchresult"]["idlist"]
        print("Found PubMed IDs:", id_list)
        if not id_list:
            raise ValueError("No IDs found for this query.")

        ids = ",".join(id_list)

        # Step 2: Fetch article summaries
        fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        fetch_params = {
            "db": "pubmed",
            "id": ids,
            "retmode": "xml"
        }
        fetch_response = requests.get(fetch_url, params=fetch_params, headers=headers, timeout=10)
        soup = BeautifulSoup(fetch_response.text, "lxml")
        articles_xml = soup.find_all("pubmedarticle")
        print("Articles found in XML:", len(articles_xml))

        articles_info = []
        for article, pmid in zip(articles_xml, id_list):
            title_tag = article.find("articletitle")
            abstract_tag = article.find("abstract")
            date_tag = article.find("pubdate")
            author_tags = article.find_all("author")

            # Title
            title = title_tag.get_text(strip=True) if title_tag else "No title"

            # Abstract
            abstract = abstract_tag.get_text(separator=" ", strip=True) if abstract_tag else "No abstract available"

            # Authors
            authors = []
            for author in author_tags:
                last = author.find("lastname")
                fore = author.find("forename")
                if last and fore:
                    authors.append(f"{fore.get_text()} {last.get_text()}")
                elif last:
                    authors.append(last.get_text())
            authors = authors if authors else ["No authors listed"]

            # Publication Date
            pub_date = "No date"
            if date_tag:
                year = date_tag.find("year")
                month = date_tag.find("month")
                pub_date = f"{month.get_text()} {year.get_text()}" if year and month else year.get_text() if year else "No date"

            # PubMed Article URL
            url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"

            print(f"Article: {title}\n   - Authors: {authors}\n   - Date: {pub_date}\n   - URL: {url}\n")
            articles_info.append({
                "title": title,
                "abstract": abstract,
                "authors": authors,
                "publication_date": pub_date,
                "article_url": url
            })

        # Mock data fallback
        if not articles_info and use_mock_if_empty:
            print("No valid articles found, returning mock data.")
            return [{
                "title": "Simulated Study on Fever",
                "abstract": "This is a simulated abstract on the treatment of fever in adults.",
                "authors": ["John Doe", "Jane Smith"],
                "publication_date": "March 2024",
                "article_url": "https://pubmed.ncbi.nlm.nih.gov/12345678/"
            }]
        return articles_info

    except Exception as e:
        print(f"Error during PubMed fetch: {e}")
        if use_mock_if_empty:
            return [{
                "title": "Simulated Study on Fever",
                "abstract": "This is a simulated abstract on the treatment of fever in adults.",
                "authors": ["John Doe", "Jane Smith"],
                "publication_date": "March 2024",
                "article_url": "https://pubmed.ncbi.nlm.nih.gov/12345678/"
            }]
        else:
            return [{"message": f"Error: {e}"}]


  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached charset_normalizer-3.4.4-cp313-cp313-win_amd64.whl.metadata (38 kB)
  Using cached urllib3-2.6.2-py3-none-any.whl.metadata (6.6 kB)
  Using cached soupsieve-2.8.1-py3-none-any.whl.metadata (4.6 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.4-cp313-cp313-win_amd64.whl (107 kB)
Using cached urllib3-2.6.2-py3-none-any.whl (131 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 2.3 MB/s

In [3]:
fetch_pubmed_articles_with_metadata("fever")

Found PubMed IDs: ['41486359', '41486132', '41485978']
Articles found in XML: 3
Article: Peripherally inserted central catheter line-induced fevers: a case report.
   - Authors: ["Catherine Van't Hoff", 'Benjamin H L Harris', 'Ahilan Kailaya-Vasan', 'Michael B Fertleman', 'Louis J Koizia']
   - Date: Jan 2026
   - URL: https://pubmed.ncbi.nlm.nih.gov/41486359/

Article: A new method to eliminate dog ears in laparoscopic low anterior resection.
   - Authors: ['Yunpeng Zhang', 'Ming Xu', 'Daqing Cheng', 'Jue Wang', 'Peng Sun']
   - Date: Jan 2026
   - URL: https://pubmed.ncbi.nlm.nih.gov/41486132/

Article: Successful therapy of a dog with trapped neutrophil syndrome using a combination of cyclosporine and prednisolone: a case report.
   - Authors: ['Ryo Miyamoto', 'Risa Kiyota', 'Tetsuro Ito', 'Hirotaka Igarashi', 'Sakurako Neo', 'Shinichiro Kimoto', 'Yoshiharu Ohsato', 'Junichi Kamiie', 'Masaharu Hisasue']
   - Date: Dec 2025
   - URL: https://pubmed.ncbi.nlm.nih.gov/41485978/



C:\Users\Nitish\AppData\Local\Temp\ipykernel_16244\3879368025.py:34: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(fetch_response.text, "lxml")


[{'title': 'Peripherally inserted central catheter line-induced fevers: a case report.',
  'abstract': 'Peripherally inserted central catheter lines are increasingly common as medical innervation advances. Peripherally inserted central catheter lines are generally well tolerated with few complications, the most common being pain, bleeding, line dislodgement or blockage. However, there are more serious complications such as line sepsis. We present a case of a 32-year-old Indian woman with recurrent fever in the context of sterile peripherally inserted central catheter lines. Noninfectious fevers can occur in patients with peripherally inserted central catheter lines with the underlying mechanism not being certain. © 2026. The Author(s).',
  'authors': ["Catherine Van't Hoff",
   'Benjamin H L Harris',
   'Ahilan Kailaya-Vasan',
   'Michael B Fertleman',
   'Louis J Koizia'],
  'publication_date': 'Jan 2026',
  'article_url': 'https://pubmed.ncbi.nlm.nih.gov/41486359/'},
 {'title': 'A ne

In [3]:
fetch_pubmed_articles_with_metadata("body pain")

Found PubMed IDs: ['41485855', '41485517', '41485027']
Articles found in XML: 3
Article: Uterine closure after cesarean delivery: surgical technique, biological rationale, and clinical implications.
   - Authors: ['Emmanuel Bujold', 'Roberto Romero']
   - Date: Jan 2026
   - URL: https://pubmed.ncbi.nlm.nih.gov/41485855/

Article: The peripheral alpha-2 antagonist vatinoxan improves dexmedetomidine-induced perivascular cerebrospinal fluid flow without affecting electroencephalogram activity in female rats.
   - Authors: ['Hanna Antila', 'Sonja C Jalonen', 'Niklas Daniel Åke Persson', 'Martta Peltoniemi', 'Terhi J Lohela', 'Tuomas O Lilius']
   - Date: Jan 2026
   - URL: https://pubmed.ncbi.nlm.nih.gov/41485517/

Article: THC induced similar physiological effects on HIV transgenic rats and their controls without affecting HIV-induced deficits in effortful motivation.
   - Authors: ['Sunitha Vemuri', 'Samantha M Ayoub', 'Arpi Minassian', 'Jared W Young']
   - Date: Jan 2026
   - URL: htt

C:\Users\Nitish\AppData\Local\Temp\ipykernel_5296\3879368025.py:34: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(fetch_response.text, "lxml")


[{'title': 'Uterine closure after cesarean delivery: surgical technique, biological rationale, and clinical implications.',
  'abstract': "Normal uterine function depends on cyclical regeneration and the capacity to sustain pregnancy. A cesarean incision represents an injury to this remarkable organ. Although the uterus possesses exceptional healing potential, cesarean delivery increases the risk of secondary infertility, pelvic pain, uterine rupture, and abnormal placentation in subsequent pregnancies. The two most important determinants of successful hysterotomy healing after cesarean delivery are the location of the incision and the surgical technique used for closure. The anatomic site of entry-whether the corpus, lower uterine segment, or cervix-defines the tissue composition, vascularity, and contractility at the wound margins, which in turn influence how the scar remodels and withstands subsequent pregnancies. Surgical technique is also important. A robust body of experimental a

In [4]:
fetch_pubmed_articles_with_metadata("headeque")

Found PubMed IDs: []
Error during PubMed fetch: No IDs found for this query.


[{'title': 'Simulated Study on Fever',
  'abstract': 'This is a simulated abstract on the treatment of fever in adults.',
  'authors': ['John Doe', 'Jane Smith'],
  'publication_date': 'March 2024',
  'article_url': 'https://pubmed.ncbi.nlm.nih.gov/12345678/'}]

In [5]:
fetch_pubmed_articles_with_metadata("headache")

Found PubMed IDs: ['41476195', '41475993', '41475886']
Articles found in XML: 3
Article: SGLT2 inhibitors prevent long-COVID-associated cognitive and pain symptoms in type 2 diabetes patients.
   - Authors: ['Han-Wei Yeh', 'Chung-Hsien Chaou', 'Shun-Fa Yang', 'Yu-Hsun Wang', 'Yu-Hsiang Kuan', 'Chao-Bin Yeh']
   - Date: Dec 2025
   - URL: https://pubmed.ncbi.nlm.nih.gov/41476195/

Article: Neurologic manifestations in Kawasaki disease and their clinical implications: a Korean nationwide pediatric cohort study.
   - Authors: ['Hee Joung Choi', 'So Hyeon Gwon', 'Nak-Hoon Son', 'Jun Chul Byun']
   - Date: Dec 2025
   - URL: https://pubmed.ncbi.nlm.nih.gov/41475993/

Article: Therapeutic effectiveness and safety profile of steroid-based greater occipital nerve blocks in cluster headaches: a systematic review and meta-analysis.
   - Authors: ['Ashmit Daiyan Mustafa', 'Luciana Freire de Oliveira', 'Raquel Espinola Saldanha', 'Fernanda Cravo Aragão', 'Karan Varshney', 'Zheng Yang Ding', 'Zamil

C:\Users\Nitish\AppData\Local\Temp\ipykernel_20668\3879368025.py:34: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(fetch_response.text, "lxml")


[{'title': 'SGLT2 inhibitors prevent long-COVID-associated cognitive and pain symptoms in type 2 diabetes patients.',
  'abstract': 'Long COVID presents significant health challenges, especially for patients with type 2 diabetes. Emerging evidence suggests that sodium-glucose cotransporter-2 (SGLT2) inhibitors may provide protective effects against COVID-19 complications, but their role in reducing long COVID risk remains unclear. Utilizing the TriNetX platform, a retrospective cohort study was conducted among adults with type 2 diabetes diagnosed with COVID-19 between January 1, 2020, and June 30, 2024. Propensity score matching balanced demographic, clinical, and comorbidity profiles between SGLT2 inhibitor users and non-users. Cox proportional hazards regression assessed the risk of long COVID, defined by a spectrum of post-COVID-19 conditions. Among 5,162 matched pairs, SGLT2 inhibitor use was associated with a significantly lower risk of long COVID (HR\u2009=\u20090.85, 95% CI: 0.

In [4]:
# symptom_extractor.py
import re

def extract_symptoms(text: str) -> list[str]:
    # Very basic extraction
    symptoms = re.findall(r"\b(headache|fever|nausea|fatigue|pain)\b", text.lower())
    return list(set(symptoms))

In [5]:
%pip install --force-reinstall pydantic-core
%pip install openai
# summarizer.py
# tools/summarizer.py
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def summarize_text(text: str) -> str:
    prompt = f"Summarize the following medical abstract:\n\n{text}"

    response = client.chat.completions.create(
        model="gpt-4",  # Or "gpt-3.5-turbo"
        messages=[
            {"role": "system", "content": "You are a medical research summarizer."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content.strip()

  Using cached pydantic_core-2.41.5-cp313-cp313-win_amd64.whl.metadata (7.4 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached pydantic_core-2.41.5-cp313-cp313-win_amd64.whl (2.0 MB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)

  Attempting uninstall: typing-extensions

    Found existing installation: typing_extensions 4.15.0

    Uninstalling typing_extensions-4.15.0:

   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
      Successfully uninstalled typing_extensions-4.15.0
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   -------------------------------

In [6]:
# diagnosis_tools.py
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_diagnosis(symptoms: list[str]) -> str:
    prompt = f"Patient has symptoms: {', '.join(symptoms)}. Suggest possible medical diagnoses.suggest me a possible cure fro the same"

    response = client.chat.completions.create(
        model="gpt-4",  # or "gpt-3.5-turbo"
        messages=[
            {"role": "system", "content": "You are a helpful medical assistant."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content.strip()


In [7]:
# fastapi_app.py
import sys
sys.path.append('..')
from fastapi import FastAPI
from pydantic import BaseModel
from tools.diagnosis_tools import get_diagnosis
from tools.symptom_extractor import extract_symptoms
from tools.pubmed_fetcher import fetch_pubmed_articles_with_metadata
from tools.summarizer import summarize_text

app = FastAPI()

class SymptomInput(BaseModel):
    description:str

@app.post("/diagnosis")
def diagnose_patient(data:SymptomInput):
    symptoms = extract_symptoms(data.description)
    diagnosis = get_diagnosis(symptoms)
    pwbmed_raw = fetch_pubmed_articles_with_metadata(" ".join(symptoms))
    summary = summarize_text(str(pwbmed_raw)[:3000])
    
    return {
        "symptom":symptoms,
        "diagnosis":diagnosis,
        "pubmed_summary" :summary
    }

In [ ]:
# mcp_tools.py
import sys
sys.path.append('..')
from fastmcp import FastMCP
from tools.diagnosis_tools import get_diagnosis

from tools.symptom_extractor import extract_symptoms
from tools.pubmed_fetcher import fetch_pubmed_articles_with_metadata
from tools.summarizer import summarize_text
mcp = FastMCP("medical diagnosis custom mcp")

@mcp.tool()
def full_medical_analysis(symptom_text):
    symptoms = extract_symptoms(symptom_text)
    diagnosis = get_diagnosis(symptoms)
    pwbmed_raw = fetch_pubmed_articles_with_metadata(" ".join(symptoms))
    summary = summarize_text(str(pwbmed_raw)[:3000])
    return {
        "symptom": symptoms,
        "diagnosis": diagnosis,
        "pubmed_summary": summary
    }

if __name__ == "__main__":
    import threading
    def _run_mcp():
        mcp.run()
    thread = threading.Thread(target=_run_mcp, daemon=True)
    thread.start()
    print("FastMCP server started in background thread.")

FastMCP server started in background thread.


[01/06/26 17:27:53] INFO     Starting server "medical diagnosis custom mcp"...                        server.py:171

Exception in thread Thread-4 (_run_mcp):
Traceback (most recent call last):
  File "C:\Users\Nitish\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 1041, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "C:\Users\Nitish\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 992, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Nitish\AppData\Local\Temp\ipykernel_20596\298369217.py", line 27, in _run_mcp
    mcp.run()
    ~~~~~~~^^
  File "d:\GenAI\29th Dec 2025-20251229T153329Z-1-001\29th Dec 2025\mcp\Lib\site-packages\fastmcp\server\server.py", line 172, in run
    anyio.run(self.run_async, transport)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\GenAI\29th Dec 2025-20251229T153329Z-1-001\29th Dec 2025\mcp\Lib\site-packages\anyio\_core\_eventloop.py", line 75, in run
    return async_backend.run(func, args, {}, backend_options)
           ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^

: 

In [ ]:
# Start FastMCP using HTTP transport (avoids notebook stdio errors)
import threading, time, sys

m = globals().get('mcp')
if m is None:
    try:
        from mcp_tools import mcp as m
    except Exception:
        from tools.mcp_tools import mcp as m

def _run_mcp():
    try:
        m.run(transport='streamable-http', show_banner=False, host='127.0.0.1', port=8000)
    except Exception as e:
        print('FastMCP run failed:', repr(e))

threading.Thread(target=_run_mcp, daemon=True).start()
time.sleep(0.2)
print('FastMCP (HTTP) start attempted on 127.0.0.1:8000')

d:\GenAI\29th Dec 2025-20251229T153329Z-1-001\29th Dec 2025\mcp\Scripts\python.exe
